# Checkpoint 6: Labels and what we know about outcomes

**Goal:** understand the answer attached to a training example and when that answer is available. Run independently with `.venv`.

Features describe information available at the prediction time. A label describes what happened afterwards. For this task, label **1** means an incident in `(decision time, decision time + 6 hours]`. Label **0** requires sufficiently complete follow-up with no incident in that interval. **Unknown** is not a third prediction class: it means the row is not yet eligible for supervised training under our policy.

At an 11 AM decision, an incident at 11 AM is excluded; an incident at 5 PM is included; one after 5 PM is excluded. These future-window boundaries are specified by the README.

There are three distinct times to track:
1. Decision time: when we would predict.
2. Incident time: when the outcome actually occurred.
3. Label availability time: when an audited report could be used for learning.

A fourth time, the **training cutoff**, asks what labels were available when fitting the model. Eventual labels can evaluate old predictions later, but cannot train a historical model before their reports arrived.


In [2]:
from pathlib import Path
from datetime import datetime, timedelta, timezone
import json
import pandas as pd
from IPython.display import display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "data/labels.jsonl").is_file()
             and (p / "src/dispatch_risk/contracts.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run from inside the candidate repository.")

def utc(text):
    value = datetime.fromisoformat(text.replace("Z", "+00:00"))
    if value.tzinfo is None:
        raise ValueError("Explicit timezone required")
    return value.astimezone(timezone.utc)

def load(name):
    with (ROOT / "data" / name).open() as handle:
        return [json.loads(line) for line in handle if line.strip()]

labels = load("labels.jsonl")
decisions = load("decision_times.jsonl")
print("Loaded", len(labels), "incident reports and", len(decisions), "decision checkpoints.")


Loaded 133 incident reports and 1800 decision checkpoints.


## 1. An incident happens before its report arrives

Teaching example: predict at 11 AM. An incident occurs at 3 PM, but its audited report arrives at 9 AM the next day.

At 6 PM on the prediction day, the six hour prediction window is over, yet this incident's label is still unavailable. Waiting for the window to finish is not by itself proof that all reports have arrived.

At 10 AM the next day, the known report establishes a positive. Its existence should not change what telemetry was available at the original 11 AM decision.


In [3]:
decision = utc("2026-01-01T11:00:00Z")
end = decision + timedelta(hours=6)
report = {"incident_id": "teaching-incident", "shipment_id": "teaching-shipment",
          "incident_at": "2026-01-01T15:00:00Z",
          "label_available_at": "2026-01-02T09:00:00Z", "severity": 1}

def within_horizon(incident_time, decision_time):
    return decision_time < incident_time <= decision_time + timedelta(hours=6)

for fit_time in (utc("2026-01-01T18:00:00Z"), utc("2026-01-02T10:00:00Z")):
    report_known = utc(report["label_available_at"]) <= fit_time
    print("Training cutoff:", fit_time.isoformat(),
          "| window ended:", end <= fit_time,
          "| report available:", report_known,
          "| known positive:", report_known and within_horizon(utc(report["incident_at"]), decision))
assert not within_horizon(decision, decision)
assert within_horizon(end, decision)
assert not within_horizon(end + timedelta(seconds=1), decision)
print("Horizon boundary checks passed.")


Training cutoff: 2026-01-01T18:00:00+00:00 | window ended: True | report available: False | known positive: False
Training cutoff: 2026-01-02T10:00:00+00:00 | window ended: True | report available: True | known positive: True
Horizon boundary checks passed.


## 2. Why negative labels need more evidence

A confirmed in-window incident proves a positive. Absence of a record proves a negative only if the records cover the entire window sufficiently completely.

For illustration, imagine operations supplies an **audit-complete-through timestamp** for this shipment. It means all incidents up to that time are fully reported, and that guarantee is known at the training cutoff. The supplied dataset does NOT contain such a field. We pass one manually only in the teaching example to make the missing information explicit.

This helper requires the full six hour window to end before admitting either class. This conservative choice avoids admitting quick positives from recent windows while their potential negatives remain immature. It is an illustrative eligibility policy, not the completed production label builder. Real evaluation will need a consistent maturity rule and a documented reporting-completeness assumption.


In [4]:
def teaching_outcome(reports, shipment, decision_time, training_cutoff,
                     audit_complete_through=None):
    """Illustrate availability and completeness; audit guarantee is hypothetical."""
    horizon_end = decision_time + timedelta(hours=6)
    if horizon_end > training_cutoff:
        return None, "Outcome window is still open"
    known = [r for r in reports if r["shipment_id"] == shipment
             and utc(r["label_available_at"]) <= training_cutoff]
    if any(within_horizon(utc(r["incident_at"]), decision_time) for r in known):
        return 1, "An available audited incident falls inside the horizon"
    if audit_complete_through is not None:
        if audit_complete_through > training_cutoff:
            raise ValueError("Audit completeness cannot extend past the training cutoff")
        if audit_complete_through >= horizon_end:
            return 0, "Complete follow-up covers the horizon and no incident was recorded"
    return None, "No known incident, but complete follow-up is unestablished"

same_day = utc("2026-01-01T18:00:00Z")
next_day = utc("2026-01-02T10:00:00Z")
examples = [
    ("Report not yet available", teaching_outcome([report], "teaching-shipment", decision, same_day)),
    ("Report available next day", teaching_outcome([report], "teaching-shipment", decision, next_day)),
    ("No report and no completeness guarantee", teaching_outcome([], "teaching-shipment", decision, next_day)),
    ("No incident and hypothetical complete audit", teaching_outcome([], "teaching-shipment", decision, next_day, end)),
]
for description, result in examples:
    print(description, "->", result)
assert [result[0] for _, result in examples] == [None, 1, None, 0]
assert teaching_outcome([], "teaching-shipment", decision, decision)[0] is None
print("Positive / negative / unknown checks passed.")


Report not yet available -> (None, 'No known incident, but complete follow-up is unestablished')
Report available next day -> (1, 'An available audited incident falls inside the horizon')
No report and no completeness guarantee -> (None, 'No known incident, but complete follow-up is unestablished')
No incident and hypothetical complete audit -> (0, 'Complete follow-up covers the horizon and no incident was recorded')
Positive / negative / unknown checks passed.


## 3. What can the actual files establish?

We can measure observed incident reporting delays and count checkpoint horizons with a listed incident. We cannot infer a universal maximum reporting delay or complete surveillance of nonincident shipments from those positive reports alone.

The following labels are **retrospective descriptions of the full supplied incident table**, not training eligibility at a historical cutoff. Rows without listed incidents remain unclassified here. Looking at reporting delays is a data audit; it must not silently set a future evaluation cutoff using final test information.


In [5]:
label_table = pd.DataFrame(labels)
label_table["incident_at_utc"] = pd.to_datetime(label_table["incident_at"], utc=True)
label_table["label_available_at_utc"] = pd.to_datetime(label_table["label_available_at"], utc=True)
label_table["report_delay_hours"] = (
    label_table["label_available_at_utc"] - label_table["incident_at_utc"]
).dt.total_seconds() / 3600
display(label_table[["shipment_id", "incident_at_utc", "label_available_at_utc", "report_delay_hours"]].head())
display(label_table["report_delay_hours"].describe().to_frame())

by_shipment = {}
for item in labels:
    by_shipment.setdefault(item["shipment_id"], []).append(item)
listed_positive = 0
for item in decisions:
    when = utc(item["decision_time"])
    listed_positive += any(within_horizon(utc(r["incident_at"]), when)
                           for r in by_shipment.get(item["shipment_id"], []))
print("Checkpoints with a listed incident in their horizon:", listed_positive)
print("Checkpoints without a listed incident (not automatically confirmed negative):", len(decisions) - listed_positive)
print("No audit-complete-through or explicit observation-end field exists in these input records.")


,shipment_id,incident_at_utc,label_available_at_utc,report_delay_hours
0,s-00000,2026-01-01 17:00:00+00:00,2026-01-02 11:00:00+00:00,18.0
1,s-00001,2026-01-01 20:01:00+00:00,2026-01-02 15:01:00+00:00,19.0
2,s-00005,2026-01-02 08:05:00+00:00,2026-01-03 07:05:00+00:00,23.0
3,s-00006,2026-01-02 11:06:00+00:00,2026-01-03 11:06:00+00:00,24.0
4,s-00009,2026-01-02 20:09:00+00:00,2026-01-03 23:09:00+00:00,27.0


,report_delay_hours
count,133.000000
mean,22.909774
std,3.112678
min,18.000000
25%,20.000000
50%,23.000000
75%,26.000000
max,28.000000


Checkpoints with a listed incident in their horizon: 140
Checkpoints without a listed incident (not automatically confirmed negative): 1660
No audit-complete-through or explicit observation-end field exists in these input records.


## 4. Decision still needed before creating the training set

| Option | Benefit | Limitation |
|---|---|---|
| Explicit audit/observation coverage from a data owner | Strong basis for negatives | Not provided; exercise allows no clarification |
| Document a closed-world, mature-dataset assumption and reporting grace period | Enables a thin implementation under an explicit assumption | Cannot prove negative completeness from these files; needs sensitivity checks and clear limits |
| Leave every unverified negative unknown | Avoids inventing negative certainty | Leaves no useful two-class training set if no coverage assumption is made |

We have NOT chosen a production grace period or claimed that the observed maximum delay is a guaranteed bound. The final policy must be documented, applied consistently around the training/evaluation split, and compatible with the public interface (which has no explicit training-cutoff argument).

Unknown rows cannot be encoded as label 0. A production implementation might omit them with accounting or reject unsupported inputs under its documented policy. We will choose this explicitly.

**Interview notes:** “I separated incident time from report availability. I required mature outcomes and documented the assumptions supporting negative labels, rather than treating missing reports as proof of no incident.”

**Try explaining this:** at 6 PM, the 11 AM prediction window has ended but the audit report arrives tomorrow. Does having no available report yet establish label zero? Why?

**Next:** choose and document a workable completeness policy and ordered by time evaluation design, then construct the first training table. Model selection follows once examples and evaluation are sound.
